# Práctica: uso de un modelo guardado con Joblib
**Gael Rodríguez Jiménez** — Analítica de datos, Tecmilenio · 24 de septiembre de 2026

Se carga `modelo_wine.pkl` y se hacen predicciones **sin volver a entrenar**.

In [1]:
# Fija scikit-learn 1.9.0: un .pkl solo es fiable con la misma versión con que se guardó.
import sys, subprocess, importlib.metadata
version = "1.9.0"
if importlib.metadata.version("scikit-learn") != version:
    if "sklearn" in sys.modules:
        raise RuntimeError("Reinicia la sesión y ejecuta esta celda antes de importar sklearn.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn==" + version])
import sklearn
print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.9.0


## Paso 1. Importar librerías

In [2]:
import joblib
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve

## Paso 2. Cargar el modelo
Si `modelo_wine.pkl` no está junto al notebook (por ejemplo en Colab), se descarga de este mismo repositorio.

In [3]:
ruta_modelo = Path("modelo_wine.pkl")
if not ruta_modelo.is_file():
    urlretrieve("https://raw.githubusercontent.com/Maade0n/analitica-datos-tecmilenio/main/Practica_Joblib/modelo_wine.pkl", ruta_modelo)
modelo = joblib.load(ruta_modelo)
print("Modelo cargado correctamente.")

Modelo cargado correctamente.


## Paso 3. Explorar el modelo

In [4]:
print("Objeto recuperado:", type(modelo).__name__)
print("Pasos:", len(modelo.steps))
for nombre, proceso in modelo.steps:
    print(f"  {nombre} -> {type(proceso).__name__}")
print("Kernel del SVM:", modelo.named_steps["svm"].kernel)
variables = list(modelo.feature_names_in_)
print("Variables esperadas:", variables)
print("Clases:", list(modelo.classes_))

Objeto recuperado: Pipeline
Pasos: 2
  scaler -> StandardScaler
  svm -> SVC
Kernel del SVM: linear
Variables esperadas: ['alcohol', 'malic_acid', 'color_intensity', 'proline']
Clases: ['Cabernet', 'Merlot', 'Pinot Noir']


Es un `Pipeline` de 2 pasos: `StandardScaler` estandariza las 4 variables con la media y desviación aprendidas, y `SVC` lineal clasifica. La ventaja es que el escalado viaja junto al clasificador y se aplica igual al predecir.

## Paso 4. Crear un nuevo vino
Mismas columnas y mismo orden que en el entrenamiento.

In [5]:
nuevo_vino = pd.DataFrame([[12.37, 1.17, 1.95, 520]], columns=variables)
nuevo_vino

,alcohol,malic_acid,color_intensity,proline
0,12.37,1.17,1.95,520


## Paso 5. Primera predicción

In [6]:
prediccion = modelo.predict(nuevo_vino)
print("Tipo de vino predicho:", prediccion[0])

Tipo de vino predicho: Merlot


## Paso 6. ¿Fue necesario volver a entrenar?
**No.** El `.pkl` ya contiene los parámetros del escalador y los vectores de soporte del SVM aprendidos en el entrenamiento.

## Paso 7. Crear múltiples observaciones

In [7]:
vinos = pd.DataFrame([
    [14.23, 1.71, 5.64, 1065],
    [12.37, 1.17, 1.95, 520],
    [13.40, 3.91, 7.30, 750],
], columns=variables)
vinos

,alcohol,malic_acid,color_intensity,proline
0,14.23,1.71,5.64,1065
1,12.37,1.17,1.95,520
2,13.40,3.91,7.30,750


## Paso 8. Predecir

In [8]:
predicciones = modelo.predict(vinos)
print("Predicciones generadas:", len(predicciones))

Predicciones generadas: 3


Se genera **una predicción por fila** porque cada fila es un vino independiente; las columnas son sus 4 características.

## Paso 9. Presentar los resultados

In [9]:
resultado = vinos.copy()
resultado["vino_predicho"] = predicciones
resultado

,alcohol,malic_acid,color_intensity,proline,vino_predicho
0,14.23,1.71,5.64,1065,Cabernet
1,12.37,1.17,1.95,520,Merlot
2,13.40,3.91,7.30,750,Pinot Noir


## Reflexión final
Fue posible predecir sin entrenar porque Joblib recuperó un Pipeline **ya ajustado**. Esto es lo que se hace en producción: se entrena una vez y el archivo se reutiliza en otra aplicación. Como estos vinos no tienen etiqueta real, no se puede medir su accuracy aquí; el desempeño (91.67 %) se midió en el notebook de construcción.